# Two-Stage INR + Graph-CNN Mesh Reconstruction from Lightcurves

sphere/ellipsoid prior (fixed topology) → **Stage 1** `LightcurveToMesh` INR (coarse mesh)
→ **Stage 2** `MeshRefinementGCN` residual refinement (`V_refined = V_inr + ΔV`)
→ differentiable cylinder gauge-fix → facet geometry (normals / areas) → convex LS+Lambert forward model
→ normalized lightcurves → MSE vs observed curves.

The GT convex hull is used as a coarse geometric prior in phase 1 (area-uniform surface
samples, never the GT mesh itself as a vertex-wise target).

**Phase 1 — INR only** (GCN parameters untouched):

$$
\mathcal{L}_1 = \mathrm{MSE}\big(\hat{y}_{\text{inr}}^{\text{al}},\, y\big) + \lambda_{\text{chamfer}}\; d_{\text{Chamfer}}\big(V_{\text{inr}},\ \partial\,\mathrm{hull}(V_{\text{GT}})\big)
$$

**Phase 2 — joint GCN + slow INR** (initialized from the best Phase-1 INR checkpoint):

$$
\mathcal{L}_2 = \mathrm{MSE}\big(\hat{y}^{\text{al}},\, y\big)
$$

Separate optimizer parameter groups with `LR_INR_STAGE2 << LR_GRAPH`.

Implementation notes

- The convex photometric forward matrix is **rebuilt from the deformed mesh every step** and remains fully differentiable through facet geometry and visibility clamps.
- The GCN operates on the fixed prior-mesh graph with sparse symmetric-normalized adjacency $\hat{A} = D^{-1/2}(A+I)D^{-1/2}$, preserving vertex count and topology (no image features, no pooling).


In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import trimesh

from hac.constants import OBSERVERS
from hac.data import get_model_folder, load_data
from hac.lc2mesh.data import create_ellipsoidal_prior_mesh
from hac.lc2mesh.forward_model import (
    AsteroidForwardModel,
    observers_to_unit_tensor,
)
from hac.lc2mesh.metrics import (
    compute_mse,
    compute_pearson_r,
    compute_per_camera_metrics,
)
from hac.lc2mesh.model import LightcurveToMesh
from hac.lc2mesh.utils import (
    chamfer_distance,
    sample_convex_hull_surface,
    set_seed,
    setup_device,
    vertex_normals_from_faces,
)
from hac.lc2mesh.visualization import (
    plot_camera_grid,
    plot_interactive_mesh,
    plot_lightcurve_comparison,
)
from hac.mesh import normalize_mesh_to_challenge_cylinder

torch.set_default_dtype(torch.float32)


In [ ]:
# ============================== CONFIG ==============================
# --- Data ---
ASTEROID_ID = 3
CURVE_TYPE = "intensity"
SEED = 42
TARGET_GPU_ID = 1
PRIOR_SHAPE_TYPE = "ellipsoid"  # "ellipsoid", "peanut", "cube", "diamond" or "irregular"

# --- Prior mesh: icosphere scaled to the challenge cylinder (R, R, 1) ---
N_SUBDIVISIONS = 4  # 4 -> 2562 vertices / 5120 faces

# --- Stage 1: INR deformer (LightcurveToMesh, coarse_deformer only) ---
INR_HIDDEN_DIM = 128
INR_NUM_LAYERS = 4
COARSE_W0 = 30.0

# --- Stage 2: graph-CNN refiner (MeshRefinementGCN) ---
GCN_HIDDEN_DIM = 128
GCN_NUM_LAYERS = 6
GCN_FINAL_INIT_SCALE = 1e-3  # near-zero output init -> refinement starts at the Stage-1 mesh

# --- Convex forward model (photometric settings follow forward_model.ipynb) ---
LAMBERT_WEIGHT = 0.5

# --- Phase 1: INR only | L1 = aligned LC MSE + hull chamfer + laplacian ---
STAGE1_STEPS = 2000
LR_INR_STAGE1 = 1e-4
LAMBDA_CHAMFER = 0.3             # GT-convex-hull Chamfer weight
HULL_N_SAMPLES = 4096            # area-uniform samples on the GT hull surface

# --- Phase 2: joint GCN + slow INR | L2 = aligned LC MSE + tether + hull chamfer ---
STAGE2_STEPS = 2000
LR_GRAPH = 1e-4
LR_INR_STAGE2 = 1e-6             # << LR_GRAPH: gentle INR fine-tuning only

PRINT_EVERY = 50

CHECKPOINT_DIR = Path("ckpt")
RECON_DIR = Path("recon")
CHECKPOINT_DIR.mkdir(exist_ok=True)
RECON_DIR.mkdir(exist_ok=True)

CHECKPOINT_PATH = CHECKPOINT_DIR / f"inr_forward_train_prior_{PRIOR_SHAPE_TYPE}_ast{ASTEROID_ID}.pth"
# =====================================================================

set_seed(SEED)
device = setup_device(TARGET_GPU_ID)


In [ ]:
# --- Observed lightcurves (the training target) ---
camera_names, e_obs = observers_to_unit_tensor(OBSERVERS, device=device, dtype=torch.float32)
e_sun = torch.tensor([-1.0, 0.0, 0.0], dtype=torch.float32, device=device)

ds_real = load_data(ASTEROID_ID, origin="real", curve_type=CURVE_TYPE)
observed = torch.as_tensor(
    ds_real.data[camera_names].to_numpy(dtype=np.float32, copy=True).T,
    dtype=torch.float32,
    device=device,
)
observed_norm = observed / observed.mean(dim=1, keepdim=True).clamp_min(1e-12)
n_phase = observed.shape[1]
phases = torch.arange(n_phase, dtype=torch.float32, device=device) * (2.0 * torch.pi / n_phase)

# --- Sphere/ellipsoid prior with fixed topology (same builder as priormesh.ipynb) ---
prior = create_ellipsoidal_prior_mesh(
    asteroid_id=ASTEROID_ID, subdivisions=N_SUBDIVISIONS, device=device, shape_type=PRIOR_SHAPE_TYPE
)
base_vertices = prior["prior_vertices_batch"].to(dtype=torch.float32)  # [1, V, 3]
faces = prior["prior_faces_t"]                                         # [F, 3]
n_vertices = int(base_vertices.shape[1])

# True vertex normals of the (fixed) prior mesh: deformation directions for the INR.
base_normals = vertex_normals_from_faces(base_vertices, faces)  # [1, V, 3]

# --- Two-stage model: Stage-1 INR (net.inr) + Stage-2 graph refiner (net.gcn).
# The mesh graph (sparse normalized adjacency) is built once from the fixed
# prior topology inside the GCN.
net = LightcurveToMesh(
    faces=faces,
    inr_hidden_dim=INR_HIDDEN_DIM,
    inr_num_layers=INR_NUM_LAYERS,
    coarse_w0=COARSE_W0,
    gcn_hidden_dim=GCN_HIDDEN_DIM,
    gcn_num_layers=GCN_NUM_LAYERS,
    gcn_final_init_scale=GCN_FINAL_INIT_SCALE,
    R=prior["R_CYLINDER"]
).to(device)
n_params_inr = sum(p.numel() for p in net.inr.parameters())
n_params_gcn = sum(p.numel() for p in net.gcn.parameters())

print(f"Asteroid {ASTEROID_ID}: cameras={len(camera_names)}, phases={n_phase}")
print(
    f"Prior mesh: {n_vertices} vertices, {int(faces.shape[0])} faces "
    f"(cylinder R={prior['R_CYLINDER']}) \n"
    f"Prior mesh extents: "
    f"x[{base_vertices[0, :, 0].min():.3f}, {base_vertices[0, :, 0].max():.3f}] "
    f"y[{base_vertices[0, :, 1].min():.3f}, {base_vertices[0, :, 1].max():.3f}] "
    f"z[{base_vertices[0, :, 2].min():.3f}, {base_vertices[0, :, 2].max():.3f}]"

)
print(f"Parameters: INR={n_params_inr:,} | GCN={n_params_gcn:,}")


In [ ]:
# --- Test the net on the prior mesh (no training yet) and visualize the output ---

net.eval()
with torch.no_grad():
    inr_init, refined_init = net(base_vertices, base_normals)
net.train()

init_vertices_np = base_vertices[0].detach().cpu().numpy()
print(f"Stage-1 INR vertices: {tuple(inr_init.shape)} | Stage-2 refined: {tuple(refined_init.shape)}")

assert inr_init.shape == base_vertices.shape
assert refined_init.shape == base_vertices.shape

fig_init = plot_interactive_mesh(
    vertices=init_vertices_np,
    faces=faces.detach().cpu().numpy(),
    title=(
        f"Asteroid {ASTEROID_ID}: prior mesh "
        f"({int(faces.shape[0])} faces)"
    ),
    mesh_color="#c0b08a",
    show_edges=False,
)
fig_init.show()

fig_deformed = plot_interactive_mesh(
    vertices=refined_init[0].detach().cpu().numpy(),
    faces=faces.detach().cpu().numpy(),
    title=(
        f"Asteroid {ASTEROID_ID}: INR+GCN output BEFORE training {int(faces.shape[0])} faces)"
    ),
    mesh_color="#c0b08a",
    show_edges=False,
)
fig_deformed.show()


In [ ]:
# --- Differentiable convex mesh -> lightcurve pipeline + loss ---
R_CYL = float(prior["R_CYLINDER"])


def facet_geometry(vertices: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Facet normals [F,3] and areas [F] from vertices [V,3] (differentiable)."""
    tri = vertices[faces]  # [F, 3, 3]
    edge1 = tri[:, 1] - tri[:, 0]
    edge2 = tri[:, 2] - tri[:, 0]
    cross = torch.cross(edge1, edge2, dim=1)
    twice_area = cross.norm(dim=1)
    normals = cross / twice_area.clamp_min(1e-12).unsqueeze(1)
    return normals, 0.5 * twice_area


def predicted_lightcurves(vertices: torch.Tensor) -> torch.Tensor:
    """Deformed mesh -> normalized convex lightcurves [C, P] (differentiable)."""
    normals, areas = facet_geometry(vertices)
    fm = AsteroidForwardModel(
        normals=normals,
        lambert_weight=LAMBERT_WEIGHT,
    )
    A = fm.build_forward_matrix(phases=phases, e_sun=e_sun, e_obs=e_obs)  # [C, P, F]
    raw = torch.einsum("cpf,f->cp", A, areas)
    return raw / raw.mean(dim=1, keepdim=True).clamp_min(1e-12)


# --- Sanity check: stage-2 gradients flow from the REFINED mesh's lightcurve
# MSE through the GCN and back into the INR ---
inr0, refined0 = net(base_vertices, base_normals)
verts0 = refined0[0]
pred0 = predicted_lightcurves(verts0)
loss0 = compute_mse(observed_norm, pred0)
loss0.backward()

grad_inr = torch.sqrt(
    sum((p.grad**2).sum() for p in net.inr.parameters() if p.grad is not None)
)
grad_gcn = torch.sqrt(
    sum((p.grad**2).sum() for p in net.gcn.parameters() if p.grad is not None)
)
all_finite = all(torch.isfinite(p.grad).all() for p in net.parameters() if p.grad is not None)
print(
    f"Sanity check: loss={float(loss0.item()):.6f} | grad_norm INR={float(grad_inr):.3e}, "
    f"GCN={float(grad_gcn):.3e} | all_finite={bool(all_finite)}"
)
print(
    f"Canonicalized extents: z[{float(verts0.detach()[:, 2].min()):.3f}, {float(verts0.detach()[:, 2].max()):.3f}], "
    f"max_r={float(verts0.detach()[:, :2].norm(dim=1).max()):.3f} (target R={R_CYL})"
)

net.zero_grad(set_to_none=True)
if device.type == "cuda":
    torch.cuda.empty_cache()


In [ ]:
# --- Convex-hull Chamfer target (used by Phase 1 ONLY) ---
gt_stl_path = get_model_folder(ASTEROID_ID) / f"asteroid{ASTEROID_ID}.stl"
hull_points = None
gt_mesh_norm = None
if gt_stl_path.exists():
    gt_mesh_norm = normalize_mesh_to_challenge_cylinder(
        trimesh.load_mesh(gt_stl_path), prior["R_CYLINDER"]
    )
    hull_samples = sample_convex_hull_surface(
        np.asarray(gt_mesh_norm.vertices), n_points=HULL_N_SAMPLES, seed=SEED
    )
    hull_points = torch.as_tensor(
        hull_samples, dtype=torch.float32, device=device
    ).unsqueeze(0)  # [1, M, 3] — batched to match [B, V, 3] predictions
    print(
        f"Hull Chamfer target: {hull_points.shape[1]} surface samples | extents "
        f"x[{hull_samples[:, 0].min():.3f}, {hull_samples[:, 0].max():.3f}] "
        f"y[{hull_samples[:, 1].min():.3f}, {hull_samples[:, 1].max():.3f}] "
        f"z[{hull_samples[:, 2].min():.3f}, {hull_samples[:, 2].max():.3f}]"
    )
else:
    print(
        f"Warning: GT STL not found at {gt_stl_path} -> Phase 1 will run with the "
        f"lightcurve loss only (no hull Chamfer)."
    )


In [ ]:
# =================== PHASE 1: train the INR ALONE ===================
optimizer = torch.optim.Adam(net.inr.parameters(), lr=LR_INR_STAGE1)
loss_fn = torch.nn.MSELoss()


history1: dict[str, list[float]] = {
    "step": [], "loss": [], "lc_mse": [], "chamfer": [], "laplacian": [], "mean_r": [], "min_r": [],
}
t_start = time.time()

for step in range(1, STAGE1_STEPS + 1):
    optimizer.zero_grad()
    vertices_inr, _ = net.inr(base_vertices, base_normals)  # Stage-1 mesh only
    pred = predicted_lightcurves(vertices_inr[0])

    lc_mse = loss_fn(observed_norm, pred)
    if hull_points is not None and LAMBDA_CHAMFER > 0.0:
        chamfer = chamfer_distance(vertices_inr, hull_points)  # batched [1,V,3] vs [1,M,3]
    else:
        chamfer = torch.zeros((), device=device)
    loss = lc_mse + LAMBDA_CHAMFER * chamfer

    loss.backward()
    torch.nn.utils.clip_grad_norm_(net.inr.parameters(), 1.0)
    optimizer.step()

    with torch.no_grad():
        r_cam = compute_pearson_r(observed_norm, pred, dim=1, eps=1e-12)

    history1["step"].append(step)
    history1["loss"].append(float(loss.item()))
    history1["lc_mse"].append(float(lc_mse.item()))
    history1["chamfer"].append(float(chamfer.item()))
    history1["mean_r"].append(float(r_cam.mean().item()))
    history1["min_r"].append(float(r_cam.min().item()))

    if step == 1 or step % PRINT_EVERY == 0:
        print(
            f"[stage 1] step {step:4d} | loss={history1['loss'][-1]:.6f} "
            f"| lc_mse={history1['lc_mse'][-1]:.6f} | chamfer={history1['chamfer'][-1]:.6f} "
            f"| mean_r={history1['mean_r'][-1]:.4f} | min_r={history1['min_r'][-1]:.4f}"
        )

In [ ]:
# --- Phase-1 result: Stage-1 INR mesh (best checkpoint) vs the GT convex hull (Chamfer target) ---
net.eval()
with torch.no_grad():
    verts_s1, _ = net.inr(base_vertices, base_normals)
net.train()

verts_s1_np = verts_s1[0].detach().cpu().numpy()
faces_np = faces.detach().cpu().numpy()

fig_s1 = plot_interactive_mesh(
    vertices=verts_s1_np,
    faces=faces_np,
    title=f"Asteroid {ASTEROID_ID}: Stage-1 INR mesh after Phase 1",
    mesh_color="#c0b08a",
    show_edges=False,
)
fig_s1.show()

if gt_mesh_norm is not None:
    hull_mesh = gt_mesh_norm.convex_hull  # same hull the Chamfer surface samples were drawn from
    fig_hull = plot_interactive_mesh(
        vertices=np.asarray(hull_mesh.vertices),
        faces=np.asarray(hull_mesh.faces),
        title=(
            f"Asteroid {ASTEROID_ID}: GT convex hull ({int(hull_mesh.faces.shape[0])} faces) "
            f"— Phase-1 Chamfer target"
        ),
        mesh_color="#a0c0ff",
        show_edges=False,
    )
    fig_hull.show()
else:
    print("GT mesh not available -> no convex hull to visualize.")


In [ ]:
# ==== PHASE 2: joint GCN + slow INR, hull-anchored aligned LC refinement ====
optimizer2 = torch.optim.Adam(
    [
        {"params": net.gcn.parameters(), "lr": LR_GRAPH},
        {"params": net.inr.parameters(), "lr": LR_INR_STAGE2},
    ]
)
loss_fn = torch.nn.MSELoss()

history2: dict[str, list[float]] = {
    "step": [], "loss": [], "mean_r": [], "min_r": [],
}
best2 = {"loss": float("inf"), "step": 0, "state": None}
t_start = time.time()

for step in range(1, STAGE2_STEPS + 1):
    optimizer2.zero_grad()
    vertices_inr, pred_vertices = net(base_vertices, base_normals)  # Stage-1 INR + Stage-2 GCN
    pred = predicted_lightcurves(pred_vertices[0])  # [C, P]

    loss = loss_fn(observed_norm, pred)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
    optimizer2.step()

    with torch.no_grad():
        r_cam = compute_pearson_r(observed_norm, pred, dim=1, eps=1e-12)

    history2["step"].append(step)
    history2["loss"].append(float(loss.item()))
    history2["mean_r"].append(float(r_cam.mean().item()))
    history2["min_r"].append(float(r_cam.min().item()))

    if history2["loss"][-1] < best2["loss"]:
        best2["loss"] = history2["loss"][-1]
        best2["step"] = step
        best2["state"] = {k: v.detach().clone() for k, v in net.state_dict().items()}

    if step == 1 or step % PRINT_EVERY == 0:
        print(
            f"[stage 2] step {step:4d} | loss={history2['loss'][-1]:.6f} "
            f"| mean_r={history2['mean_r'][-1]:.4f} | min_r={history2['min_r'][-1]:.4f}"
        )

# Keep the best joint state, not the last iterate.
net.load_state_dict(best2["state"])
torch.save(
    {
        "state_dict": best2["state"],
        "model_state": best2["state"],
        "config": {
            "asteroid_id": ASTEROID_ID,
            "n_subdivisions": N_SUBDIVISIONS,
            "inr_hidden_dim": INR_HIDDEN_DIM,
            "inr_num_layers": INR_NUM_LAYERS,
            "coarse_w0": COARSE_W0,
            "gcn_hidden_dim": GCN_HIDDEN_DIM,
            "gcn_num_layers": GCN_NUM_LAYERS,
            "gcn_final_init_scale": GCN_FINAL_INIT_SCALE,
            "lambert_weight": LAMBERT_WEIGHT,
            "lambda_chamfer": LAMBDA_CHAMFER,
            "lr_inr_stage1": LR_INR_STAGE1,
            "lr_inr_stage2": LR_INR_STAGE2,
            "lr_graph": LR_GRAPH,
            "stage1_steps": STAGE1_STEPS,
            "stage2_steps": STAGE2_STEPS,
        },
        "best_step": best2["step"],
        "best_loss": best2["loss"],
    },
    CHECKPOINT_PATH,
)
print(
    f"Stage 2 done in {(time.time() - t_start) / 60.0:.1f} min | "
    f"best loss={best2['loss']:.6f} @ step {best2['step']} | joint checkpoint: {CHECKPOINT_PATH}"
)


In [ ]:
# --- Training curves across both phases (stage 2 offset by STAGE1_STEPS) ---
s2_steps = [STAGE1_STEPS + s for s in history2["step"]]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history1["step"], history1["loss"], label="Stage 1 total", color="#d95f02")
if any(c > 0 for c in history1["chamfer"]):
    axes[0].plot(history1["step"], history1["chamfer"], label="Stage 1 hull Chamfer", color="#66a61e", alpha=0.7)
axes[0].plot(s2_steps, history2["loss"], label="Stage 2 total", color="#e7298a")
axes[0].axvline(STAGE1_STEPS, color="k", ls="--", lw=1, alpha=0.5)
axes[0].set_yscale("log")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Two-phase training losses")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history1["step"], history1["mean_r"], label="Mean r (stage 1)", color="#d95f02")
axes[1].plot(history1["step"], history1["min_r"], label="Min r (stage 1)", color="#1b9e77")
axes[1].plot(s2_steps, history2["mean_r"], label="Mean r (stage 2)", color="#e7298a")
axes[1].plot(s2_steps, history2["min_r"], label="Min r (stage 2)", color="#66a61e")
axes[1].axvline(STAGE1_STEPS, color="k", ls="--", lw=1, alpha=0.5)
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Pearson r")
axes[1].set_title("Lightcurve Correlation (aligned)")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.tight_layout()
plt.show()


In [ ]:
# --- Final evaluation: the REFINED mesh (canonical cylinder frame, aligned curves, as in training) ---
with torch.no_grad():
    inr_final, deformed_final = net(base_vertices, base_normals)
    pred_norm = predicted_lightcurves(deformed_final[0])
    pred_inr_norm = predicted_lightcurves(inr_final[0])

per_camera_df = compute_per_camera_metrics(observed_norm, pred_norm, camera_names)
per_camera_inr_df = compute_per_camera_metrics(observed_norm, pred_inr_norm, camera_names)
print(
    f"Refined mesh fit: mean_r={per_camera_df['Pearson_R'].mean():.4f}, "
    f"min_r={per_camera_df['Pearson_R'].min():.4f}, "
    f"mean MSE={per_camera_df['MSE'].mean():.6f}"
)
print(
    f"Stage-1 INR mesh fit (reference): mean_r={per_camera_inr_df['Pearson_R'].mean():.4f}, "
    f"min_r={per_camera_inr_df['Pearson_R'].min():.4f}, "
    f"mean MSE={per_camera_inr_df['MSE'].mean():.6f}"
)
display(per_camera_df.sort_values("Pearson_R"))

target_cam = "135_hor1"
fig, _ = plot_lightcurve_comparison(
    obs_norm=observed_norm.detach().cpu().numpy(),
    pred_norm=pred_norm.detach().cpu().numpy(),
    camera_keys=camera_names,
    target_cam_id=target_cam,
)
plt.show()

phase_deg = np.rad2deg(phases.detach().cpu().numpy())
fig, _ = plot_camera_grid(observed_norm, pred_norm, camera_names, phase_axis=phase_deg)
plt.show()


In [ ]:
GT_EXIST = True
model_dir = get_model_folder(ASTEROID_ID)
stl_file = model_dir / f"asteroid{ASTEROID_ID}.stl"
if not stl_file.exists():
    GT_EXIST = False
    print(f"Warning: GT STL file not found at {stl_file} -> skipping voxel measure calculation.")
else:
    # Raw challenge STLs span [-3, 3]^3, but the whole training pipeline lives in
    # the normalized cylinder frame D(0, R) x [-1, 1]. Bring GT into the same
    # frame before comparing extents or computing the voxel measure.
    true_mesh = normalize_mesh_to_challenge_cylinder(
        trimesh.load_mesh(stl_file), prior["R_CYLINDER"]
    )

# --- Interactive 3D view of the refined mesh (same utility as forward_model.ipynb) ---
vertices_np = deformed_final[0].detach().cpu().numpy()
faces_np = faces.detach().cpu().numpy()

fig_mesh = plot_interactive_mesh(
    vertices=vertices_np,
    faces=faces_np,
    title=(
        f"Asteroid {ASTEROID_ID}: INR+GCN refined mesh ({faces_np.shape[0]} faces) | "
        f"mean_r={per_camera_df['Pearson_R'].mean():.3f}"
    ),
    mesh_color="#c0b08a",
    show_edges=False,
)
fig_mesh.show()

if GT_EXIST:
    gt_fig_mesh = plot_interactive_mesh(
        vertices=true_mesh.vertices,
        faces=true_mesh.faces,
        title=f"Asteroid {ASTEROID_ID}: GT mesh, cylinder-normalized ({true_mesh.faces.shape[0]} faces)",
        mesh_color="#a0c0ff",
        show_edges=False,
    )
    gt_fig_mesh.show()

print(
    f"Refined mesh extents: "
    f"x[{vertices_np[:, 0].min():.3f}, {vertices_np[:, 0].max():.3f}] "
    f"y[{vertices_np[:, 1].min():.3f}, {vertices_np[:, 1].max():.3f}] "
    f"z[{vertices_np[:, 2].min():.3f}, {vertices_np[:, 2].max():.3f}] "
    f"(cylinder bound: R={prior['R_CYLINDER']}, z in [-1, 1])"
)
if GT_EXIST:
    print(
        f"GT mesh extents (cylinder-normalized): "
        f"x[{true_mesh.vertices[:, 0].min():.3f}, {true_mesh.vertices[:, 0].max():.3f}] "
        f"y[{true_mesh.vertices[:, 1].min():.3f}, {true_mesh.vertices[:, 1].max():.3f}] "
        f"z[{true_mesh.vertices[:, 2].min():.3f}, {true_mesh.vertices[:, 2].max():.3f}]"
    )

# --- Save the final refined mesh as STL ---
recon_mesh = trimesh.Trimesh(vertices=vertices_np, faces=faces_np, process=False)
recon_mesh.export(RECON_DIR / f"prior_{PRIOR_SHAPE_TYPE}_asteroid{ASTEROID_ID}.stl")
print(f"Saved refined mesh as STL: {RECON_DIR / f'prior_{PRIOR_SHAPE_TYPE}_asteroid{ASTEROID_ID}.stl'}")


In [ ]:
# --- Evaluation voxel measure calculation ---
from hac.eval import calculate_voxel_measure

if GT_EXIST:
    # compute the voxel measure between the predicted mesh and the ground truth mesh
    pred_mesh = trimesh.Trimesh(vertices=vertices_np, faces=faces_np, process=False)
    score = calculate_voxel_measure(true_mesh, pred_mesh)
    print(f"Voxel measure between predicted mesh and GT mesh: {score:.6f}")
else:
    print("GT mesh not available -> skipping voxel measure calculation.")